In [1]:
import pandas as pd
import polars as pl
import networkx as nx
from maplib import Model, Prefix, Variable, Template, Parameter, RDFType, Triple, a

## Main data transformation:
- Ingests gtfs transit files
- Creates RDF Node templates for each object type
- Constructs queryable graph model
- serializes to N-triple file for export

In [2]:
# Set up base identifiers for nodes (RDF standard)

BASE_STR = "http://imt542.org/transit-lab/"
GTFS_STR = "http://vocab.gtfs.org/terms#"
GEO_STR  = "http://www.w3.org/2003/01/geo/wgs84_pos#"
XSD_STR  = "http://www.w3.org/2001/XMLSchema#"

In [3]:
# Create maplib Prefix objects from base IRIs
base = Prefix(BASE_STR)
gtfs = Prefix(GTFS_STR)
geo = Prefix(GEO_STR)
xsd = Prefix(XSD_STR)

In [4]:
# Load gtfs data from data source
kcm_stops = pd.read_csv("data/kc_metro/stops.txt", dtype={"stop_id": str})
kcm_trips = pd.read_csv("data/kc_metro/trips.txt", dtype={"trip_id": str, "route_id": str})
kcm_routes = pd.read_csv("data/kc_metro/routes.txt", dtype={"route_id": str})
kcm_stop_times = pd.read_csv("data/kc_metro/stop_times.txt", dtype={"trip_id": str, "stop_id": str, "stop_headsign": str})

In [5]:
# function to build IRIs for node construction
def make_uri(prefix,series):
    return prefix + series

In [ ]:
# Create polars object from pandas dataframe and reformats to align with RDF objects. This is compatible with maplib/rust
routes_pl = pl.from_pandas(pd.DataFrame({
    "route": make_uri(BASE_STR + "route/", kcm_routes["route_id"]),
    "route_short_name": kcm_routes["route_short_name"].values,
    "description": kcm_routes["route_desc"].values,
}))

In [ ]:
stops_pl = pl.from_pandas(pd.DataFrame({
    "stop": make_uri(BASE_STR + "stop/", kcm_stops["stop_id"]),
    "stop_name": kcm_stops["stop_name"].values,
    "stop_lat": kcm_stops["stop_lat"].astype(float).values,
    "stop_lon": kcm_stops["stop_lon"].astype(float).values,
})).with_columns([
    pl.col("stop_lat").cast(pl.Decimal(precision=38, scale=12)),
    pl.col("stop_lon").cast(pl.Decimal(precision=38, scale=12)),
])

In [8]:
trips_pl = pl.from_pandas(pd.DataFrame({
    "trip": make_uri(BASE_STR + "trip/", kcm_trips["trip_id"]),
    "route": make_uri(BASE_STR + "route/", kcm_trips["route_id"]),
    "trip_headsign": kcm_trips["trip_headsign"].values,
}))


In [9]:
st_pl = pl.from_pandas(pd.DataFrame({
    "stoptime": make_uri(BASE_STR + "stoptime/", kcm_stop_times["stop_id"]),
    "trip": make_uri(BASE_STR + "trip/", kcm_stop_times["trip_id"]),
    "stop": make_uri(BASE_STR + "stop/", kcm_stop_times["stop_id"]),
    "stop_sequence": kcm_stop_times["stop_sequence"].astype(str).values,
    "arrival_time": kcm_stop_times["arrival_time"].astype(str).values,
    "departure_time": kcm_stop_times["departure_time"].astype(str).values,
}))

In [10]:
# Initiates base model for building the graph in memory. Sets up variables for each node type
m = Model()
r_var = Variable("route")
s_var = Variable("stop")
t_var = Variable("trip")
st_var = Variable("stoptime")


In [11]:
# OTTR templates for each node type.

# template has a signature and a pattern.
# Signature contains the identifier and it's parameters
# Pattern contains instances and can use parameters as arguments
route_template = Template(
    iri = base.suf("RouteTemplate"),
    parameters = [
        Parameter(variable = r_var, rdf_type=RDFType.IRI),
        Parameter(variable = Variable("route_short_name"), rdf_type=RDFType.Literal(f"{XSD_STR}string")),
        Parameter(variable = Variable("description"), rdf_type=RDFType.Literal(f"{XSD_STR}string")),
    ],
    instances=[
        Triple(r_var, a, gtfs.suf("route")), # a refers to RDFType, imported from maplib this means ?route  rdf:type  gtfs:Route
        Triple(r_var, gtfs.suf("route_short_name"), Variable("route_short_name")),
        Triple(r_var, gtfs.suf("description"), Variable("description")),
    ]
)

In [12]:
stop_template = Template(
    iri=base.suf("StopTemplate"),
    parameters=[
        Parameter(variable=s_var,                  rdf_type=RDFType.IRI),
        Parameter(variable=Variable("stop_name"),  rdf_type=RDFType.Literal(f"{XSD_STR}string")),
        Parameter(variable=Variable("stop_lat"),   rdf_type=RDFType.Literal(f"{XSD_STR}decimal")),
        Parameter(variable=Variable("stop_lon"),   rdf_type=RDFType.Literal(f"{XSD_STR}decimal")),
    ],
    instances=[
        Triple(s_var, a,                  gtfs.suf("Stop")),
        Triple(s_var, gtfs.suf("name"),   Variable("stop_name")),
        Triple(s_var, geo.suf("lat"),     Variable("stop_lat")),
        Triple(s_var, geo.suf("long"),    Variable("stop_lon")),
    ]
)

In [13]:
trip_template = Template(
    iri = base.suf("TripTemplate"),
    parameters = [
        Parameter(variable = t_var, rdf_type=RDFType.IRI),
        Parameter(variable=r_var, rdf_type=RDFType.IRI),
        Parameter(variable=Variable("trip_headsign"), rdf_type=RDFType.Literal(f"{XSD_STR}string")),
    ],
    instances=[
        Triple(t_var, a, gtfs.suf("trip")),
        Triple(t_var, gtfs.suf("route"), r_var),
        Triple(t_var, gtfs.suf("trip_headsign"), Variable("trip_headsign")),
        Triple(r_var, gtfs.suf("trip"), t_var) #backlink to route
    ]
)

In [14]:
stoptime_template = Template(
    iri = base.suf("StopTimeTemplate"),
    parameters = [
        Parameter(variable = st_var, rdf_type=RDFType.IRI),
        Parameter(variable=t_var, rdf_type=RDFType.IRI),
        Parameter(variable=s_var, rdf_type=RDFType.IRI),
        Parameter(variable = Variable("stop_sequence"), rdf_type=RDFType.Literal(f"{XSD_STR}string")),
        Parameter(variable= Variable("arrival_time"), rdf_type=RDFType.Literal(f"{XSD_STR}string")),
        Parameter(variable=Variable("departure_time"), rdf_type=RDFType.Literal(f"{XSD_STR}string")),
    ],
    instances=[
        Triple(st_var, a, gtfs.suf("stop_time")),
        Triple(st_var, gtfs.suf("trip"), t_var),
        Triple(st_var, gtfs.suf("stop"), s_var),
        Triple(st_var, gtfs.suf("stop_sequence"), Variable("stop_sequence")),
        Triple(st_var, gtfs.suf("arrival_time"), Variable("arrival_time")),
        Triple(st_var, gtfs.suf("departure_time"), Variable("departure_time")),
        Triple(t_var, gtfs.suf("stop_time"), st_var),
    ]
)

In [15]:
# Graph built out mapping the polars data objects to the OTTR templates
m.map(route_template, routes_pl)
m.map(stop_template, stops_pl)
m.map(trip_template, trips_pl)
m.map(stoptime_template, st_pl)


In [34]:
# Serializes graph model to an N-Triple file (RDF compliant)
m.write(file_path="output/transit_ntriples.nt", format="ntriples")

_Process to create .gexf file for use with visualization software like Gephi (this can be skipped if you are only interested in the graph model)_

In [25]:


G = nx.DiGraph()

visual = {
    "route":    {"color": "#e74c3c", "size": 30},
    "trip":     {"color": "#3498db", "size": 15},
    "stop_time": {"color": "#f39c12", "size": 5},
    "stop":     {"color": "#2ecc71", "size": 10},
}


In [26]:
edges = m.query(f"""
    PREFIX gtfs: <{GTFS_STR}>
    SELECT ?s ?sType ?o ?oType ?p WHERE {{
        {{
            ?s a gtfs:route .    BIND("route"    AS ?sType)
            ?s gtfs:trip ?o .    BIND("trip"     AS ?oType)
            BIND(gtfs:trip AS ?p)
        }}
        UNION
        {{
            ?s a gtfs:trip .     BIND("trip"     AS ?sType)
            ?s gtfs:stop_time ?o . BIND("stop_time" AS ?oType)
            BIND(gtfs:stop_time AS ?p)
        }}
        UNION
        {{
            ?s a gtfs:stop_time . BIND("stop_time" AS ?sType)
            ?s gtfs:stop ?o .    BIND("stop"     AS ?oType)
            BIND(gtfs:stop AS ?p)
        }}
    }}
""")

In [27]:
for row in edges.iter_rows(named=True):
    s      = row["s"]
    o      = row["o"]
    p      = row["p"].split("#")[-1]
    s_type = row["sType"]
    o_type = row["oType"]

    if s not in G:
        G.add_node(s, label=s.split("/")[-1],
            nodeType=s_type,
            color=visual[s_type]["color"],
            size=float(visual[s_type]["size"]))
    if o not in G:
        G.add_node(o, label=o.split("/")[-1],
            nodeType=o_type,
            color=visual[o_type]["color"],
            size=float(visual[o_type]["size"]))
    G.add_edge(s, o, label=p)

print(f"Graph — Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")


Graph — Nodes: 46042, Edges: 1126888


In [35]:
nx.write_gexf(G, "output/transit.gexf")

_Sample queries and outputs_

In [32]:
# Sample: get all information from a stop
print(m.query(f"""
    PREFIX gtfs: <{GTFS_STR}>
    PREFIX geo:  <http://www.w3.org/2003/01/geo/wgs84_pos#>
    SELECT ?s ?stop_name ?lat ?lon WHERE {{
        ?s a gtfs:Stop ;
           gtfs:name ?stop_name ;
           geo:lat   ?lat ;
           geo:long  ?lon .
    }} LIMIT 5
"""))

shape: (5, 4)
┌─────────────────────────────────┬──────────────────────────┬─────────────────┬───────────────────┐
│ s                               ┆ stop_name                ┆ lat             ┆ lon               │
│ ---                             ┆ ---                      ┆ ---             ┆ ---               │
│ str                             ┆ str                      ┆ decimal[38,12]  ┆ decimal[38,12]    │
╞═════════════════════════════════╪══════════════════════════╪═════════════════╪═══════════════════╡
│ <http://imt542.org/transit-lab… ┆ 1st Ave & Spring St      ┆ 47.605136900000 ┆ -122.336533000000 │
│ <http://imt542.org/transit-lab… ┆ 40th Ave NE & NE 51st St ┆ 47.665885900000 ┆ -122.284897000000 │
│ <http://imt542.org/transit-lab… ┆ NE 55th St & 39th Ave NE ┆ 47.668579100000 ┆ -122.285667000000 │
│ <http://imt542.org/transit-lab… ┆ NE 55th St & 37th Ave NE ┆ 47.668579100000 ┆ -122.288300000000 │
│ <http://imt542.org/transit-lab… ┆ NE 55th St & 35th Ave NE ┆ 47.66857910000

In [33]:
# Sample: get count of stops by geographic bucket
print(m.query(f"""
    PREFIX gtfs: <{GTFS_STR}>
    PREFIX geo:  <http://www.w3.org/2003/01/geo/wgs84_pos#>
    SELECT ?latBucket ?lonBucket (COUNT(DISTINCT ?stop) AS ?stopCount)
    WHERE {{
        {{
            SELECT ?stop
                   (ROUND(?stop_lat * 10) / 10 AS ?latBucket)
                   (ROUND(?stop_lon * 10) / 10 AS ?lonBucket)
            WHERE {{
                ?stop a gtfs:Stop ;
                      geo:lat  ?stop_lat ;
                      geo:long ?stop_lon .
            }}
        }}
    }}
    GROUP BY ?latBucket ?lonBucket
    ORDER BY DESC(?stopCount)
"""))

shape: (42, 3)
┌───────────┬───────────┬───────────┐
│ latBucket ┆ lonBucket ┆ stopCount │
│ ---       ┆ ---       ┆ ---       │
│ f64       ┆ f64       ┆ u32       │
╞═══════════╪═══════════╪═══════════╡
│ 47.6      ┆ -122.3    ┆ 874       │
│ 47.7      ┆ -122.3    ┆ 653       │
│ 47.5      ┆ -122.3    ┆ 551       │
│ 47.6      ┆ -122.4    ┆ 377       │
│ 47.7      ┆ -122.2    ┆ 362       │
│ …         ┆ …         ┆ …         │
│ 47.4      ┆ -122.4    ┆ 4         │
│ 47.9      ┆ -122.0    ┆ 3         │
│ 47.5      ┆ -121.7    ┆ 2         │
│ 47.6      ┆ -121.8    ┆ 2         │
│ 47.3      ┆ -122.5    ┆ 1         │
└───────────┴───────────┴───────────┘
